# Patient-Independent EEG Seizure Classification
## Experiment 2 — Patient-Disjoint Model Selection and Held-Out Patient Evaluation

## 1. Introduction

The one-epoch LOSO experiment provided a broad patient-level comparison between raw EEG and spectral bandpower representations, but it did not optimize training duration for either model.

This experiment therefore uses a separate **patient-disjoint model-development procedure**. Training duration is selected using four held-out validation patients, after which each model is reinitialized and retrained on the full 18-patient development cohort. The resulting models are then evaluated descriptively on three additional held-out CHB-MIT patients.

## 2. Experimental Setup

### 2.1 Runtime and Dependencies

- **MNE** — reads and filters EDF EEG recordings used for the additional held-out patients.
- **NumPy / Pandas** — manage EEG arrays, validation curves, and evaluation results.
- **PyTorch** — defines, trains, saves, and evaluates the neural-network models.
- **scikit-learn** — calculates patient-level AUROC and AUPRC.
- **SciPy** — computes Welch power spectral density and integrates spectral bandpower.
- **Matplotlib** — generates the final validation and performance figures.
- **Path** — manages dataset, cache, model, and results paths.
- **random** — controls the reproducible patient split and model-training seed.
- **re** — parses seizure annotations from CHB-MIT summary files.
- **gc / ctypes** — help release large CPU-memory allocations during training.
- Training requires a **CUDA-enabled GPU** because the raw EEG arrays are large and the final models train for multiple epochs.

In [5]:
%pip -q install mne

from google.colab import drive

import gc
import ctypes
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import mne

from scipy.signal import welch
from scipy.integrate import trapezoid
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import TensorDataset, DataLoader

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2.2 Patient-Disjoint Development Split

- The same **18 development patients** from Experiment 1 are used for model development.
- A fixed seed of **42** selects four patients for validation: `chb04`, `chb01`, `chb09`, and `chb23`.
- The remaining **14 patients** are used for training during epoch selection.
- Training and validation patients are completely disjoint.
- Both RAW and PSD use the **same split**, allowing training duration to be selected under the same patient-level evaluation framework.
- Models are allowed to train for at most **20 epochs**.

In [6]:
SEED = 42
BATCH_SIZE = 128
MAX_EPOCHS = 20

DEV_SUBJECTS = [
    "chb01", "chb02", "chb03", "chb04",
    "chb05", "chb06", "chb07", "chb08",
    "chb09", "chb10", "chb11", "chb13",
    "chb14", "chb17", "chb19", "chb20",
    "chb22", "chb23",
]

VAL_SUBJECTS = [
    "chb04",
    "chb01",
    "chb09",
    "chb23",
]

assert (
    random.Random(SEED).sample(
        DEV_SUBJECTS,
        4
    )
    == VAL_SUBJECTS
)

TRAIN_SUBJECTS = [
    subject
    for subject in DEV_SUBJECTS
    if subject not in VAL_SUBJECTS
]

assert set(TRAIN_SUBJECTS).isdisjoint(
    VAL_SUBJECTS
)

## Natural-prevalence, memory-safe training data


In [7]:

def build_training_memory_safe(subjects, representation):
    """
    Use ALL eligible windows from the supplied patients.

    Pass 1: fit normalization statistics on those patients only.
    Pass 2: allocate the final matrix once, fill patient-by-patient,
            and normalize in place to avoid multi-GB temporary copies.
    """
    feature_sum = None
    feature_sq_sum = None
    total_stat_count = 0
    total_windows = 0
    total_pos = 0
    total_neg = 0
    sample_shape = None

    for subject in subjects:
        X, y = load_subject(subject, representation)

        if sample_shape is None:
            sample_shape = X.shape[1:]
            feature_dim = X.shape[1]
            feature_sum = np.zeros(feature_dim, dtype=np.float64)
            feature_sq_sum = np.zeros(feature_dim, dtype=np.float64)

        if representation == "raw":
            for ch in range(X.shape[1]):
                xc = X[:, ch, :].astype(np.float64)
                feature_sum[ch] += xc.sum()
                feature_sq_sum[ch] += np.square(xc).sum()
                del xc
            total_stat_count += X.shape[0] * X.shape[2]
        else:
            X64 = X.astype(np.float64)
            feature_sum += X64.sum(axis=0)
            feature_sq_sum += np.square(X64).sum(axis=0)
            total_stat_count += X.shape[0]
            del X64

        total_windows += len(y)
        total_pos += int((y == 1).sum())
        total_neg += int((y == 0).sum())
        del X, y
        gc.collect()

    mean = feature_sum / total_stat_count
    variance = feature_sq_sum / total_stat_count - mean ** 2
    std = np.sqrt(np.maximum(variance, 1e-8))
    mean = mean.astype(np.float32)
    std = std.astype(np.float32)

    X_all = np.empty((total_windows, *sample_shape), dtype=np.float32)
    y_all = np.empty(total_windows, dtype=np.float32)

    offset = 0
    for subject in subjects:
        X, y = load_subject(subject, representation)
        n = len(y)
        X_all[offset:offset+n] = X
        y_all[offset:offset+n] = y
        offset += n
        del X, y
        gc.collect()

    if representation == "raw":
        for ch in range(X_all.shape[1]):
            X_all[:, ch, :] -= mean[ch]
            X_all[:, ch, :] /= (std[ch] + 1e-6)
    else:
        X_all -= mean
        X_all /= (std + 1e-6)

    print(
        f"{representation.upper()} | {total_windows} windows | "
        f"seizure={total_pos} | normal={total_neg} | "
        f"neg:pos={total_neg / total_pos:.1f}:1"
    )
    return X_all, y_all, mean, std


### 2.3 Data, Cache, and Output Paths

- `LOSO_ROOT` contains the preprocessed raw EEG patient arrays created in Experiment 1.
- `PSD_ROOT` contains the corresponding 90-feature spectral representations.
- `DATA_ROOT` points to the original CHB-MIT EDF recordings and summary files used later for the additional held-out patients.
- `OUT_DIR` stores validation curves, trained model checkpoints, and final evaluation results.
- `EXT_CACHE` stores processed held-out raw EEG arrays so EDF extraction does not need to be repeated.

In [8]:
LOSO_ROOT = Path(
    "/content/drive/MyDrive/chbmit-seizure-detection/"
    "loso_subject_data"
)

PSD_ROOT = Path(
    "/content/drive/MyDrive/chbmit-seizure-detection/"
    "loso_psd_data"
)

DATA_ROOT = Path(
    "/content/drive/MyDrive/chbmit_subset"
)

OUT_DIR = Path(
    "/content/drive/MyDrive/chbmit-seizure-detection/"
    "final_fast_model"
)

EXT_CACHE = (
    OUT_DIR
    / "external_raw_cache"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXT_CACHE.mkdir(
    parents=True,
    exist_ok=True
)

### 2.4 Compute Device

- PyTorch uses a **CUDA-enabled GPU** for model training.
- Unlike Notebook 1's CPU fallback, this experiment explicitly requires GPU execution because repeated multi-epoch training on the full raw EEG arrays is substantially more computationally expensive.
- Execution stops if a CUDA device is unavailable so the experiment is not accidentally run under an impractical CPU configuration.

In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if device.type != "cuda":
    raise RuntimeError(
        "Switch Colab to a GPU runtime before training."
    )

print("Device:", device)
print("Training patients:", TRAIN_SUBJECTS)
print("Validation patients:", VAL_SUBJECTS)

RuntimeError: Switch Colab to a GPU runtime before training.

## 3. Models and Shared Training Utilities

### 3.1 Reproducibility and Patient Data Loading

- The random seed is reset before model-development runs so NumPy and PyTorch behavior is reproducible.
- Patient data are loaded from the preprocessed files created earlier rather than reprocessing EDF recordings during model development.
- RAW and PSD use the same patient labels and differ only in the representation loaded from disk.
- Raw EEG arrays have shape **[windows, 18 channels, 1,024 samples]**, while PSD arrays contain **90 features per window**.

In [ ]:
def set_seed(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_subject(
    subject,
    representation
):

    root = (
        LOSO_ROOT
        if representation == "raw"
        else PSD_ROOT
    )

    with np.load(
        root / f"{subject}.npz"
    ) as data:

        X = data["X"].astype(np.float32)
        y = data["y"].astype(np.float32)

    return X, y

### 3.2 Model Architectures

Two different neural-network architectures are used because the RAW and PSD representations have different structures.

#### Raw EEG CNN

- The RAW model operates directly on the **18-channel EEG waveform**.
- Three 1D convolutional layers learn local temporal features from the signal.
- Feature depth increases from **18 → 32 → 64 → 128** channels.
- Max pooling reduces the temporal dimension after the first two convolutional layers.
- Adaptive average pooling compresses each feature map to a single value.
- Dropout (`0.3`) regularizes the final classifier before producing one seizure logit.

#### PSD MLP

- The PSD model receives a fixed **90-dimensional bandpower feature vector**.
- Two fully connected hidden layers reduce the representation from **90 → 64 → 32** features.
- ReLU introduces nonlinearity and dropout (`0.3`) is applied after each hidden layer.
- The final linear layer outputs one seizure logit.

In [ ]:
class RawCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv1d(
                18,
                32,
                kernel_size=7,
                padding=3
            ),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(
                32,
                64,
                kernel_size=5,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):

        return (
            self.net(x)
            .squeeze(1)
        )


class PSDMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(90, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        return (
            self.net(x)
            .squeeze(1)
        )

### 3.3 Shared Model, Loader, and Loss Helpers

- `make_model()` creates the appropriate architecture for the requested representation and moves it to the GPU.
- `DataLoader` batches windows in groups of **128** and shuffles only training data.
- A fixed PyTorch generator seed makes the training-data shuffle reproducible.
- Class imbalance is handled with `BCEWithLogitsLoss`, using a positive-class weight equal to **negative windows / positive windows** from the current training set.

In [ ]:
def make_model(representation):

    model = (
        RawCNN()
        if representation == "raw"
        else PSDMLP()
    )

    return model.to(device)


def make_loader(
    X,
    y,
    shuffle
):

    generator = torch.Generator()
    generator.manual_seed(SEED)

    dataset = TensorDataset(
        torch.from_numpy(X),
        torch.from_numpy(
            y.astype(np.float32)
        )
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=(
            generator
            if shuffle
            else None
        )
    )


def make_loss(y):

    n_positive = float(
        (y == 1).sum()
    )

    n_negative = float(
        (y == 0).sum()
    )

    pos_weight = (
        n_negative
        / n_positive
    )

    return nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [pos_weight],
            dtype=torch.float32,
            device=device
        )
    )

### 3.4 Training, Prediction, and Scaling Utilities

- `train_epoch()` performs one full pass through the training data and returns the average loss across all windows.
- During evaluation, `model.eval()` and `torch.no_grad()` disable training-specific behavior and gradient tracking.
- Model logits are converted to seizure probabilities using the **sigmoid** function.
- `scale_array()` applies the normalization statistics learned from the training patients to either RAW or PSD inputs.
- RAW EEG is standardized **per channel**, while PSD is standardized **per spectral feature**.

In [ ]:
def train_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0.0
    total_n = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += (
            float(loss.item())
            * len(y_batch)
        )

        total_n += len(y_batch)

    return (
        total_loss
        / total_n
    )


def predict(
    model,
    X,
    y
):

    loader = make_loader(
        X,
        y,
        shuffle=False
    )

    labels = []
    probabilities = []

    model.eval()

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)

            probs = torch.sigmoid(
                model(X_batch)
            ).cpu().numpy()

            labels.append(
                y_batch.numpy()
            )

            probabilities.append(
                probs
            )

    return (
        np.concatenate(labels),
        np.concatenate(probabilities)
    )


def scale_array(
    X,
    mean,
    std,
    representation
):

    if representation == "raw":

        return (
            (
                X
                - mean[None, :, None]
            )
            / (
                std[None, :, None]
                + 1e-6
            )
        ).astype(np.float32)

    return (
        (
            X - mean
        )
        / (
            std + 1e-6
        )
    ).astype(np.float32)

## 4. Natural-Prevalence Training Data

### 4.1 Memory-Safe Training Set Construction

- All eligible windows from the supplied training patients are retained at their **natural seizure prevalence**; no undersampling or oversampling is performed.
- The function uses a **two-pass procedure** to reduce memory overhead.
- **Pass 1** computes normalization statistics and counts the number of windows without concatenating every patient array.
- **Pass 2** allocates the final training arrays once, fills them patient-by-patient, and then normalizes them in place.
- For RAW EEG, normalization statistics are computed separately for each of the **18 channels**.
- For PSD, normalization statistics are computed separately for each of the **90 spectral features**.
- Because this function is called with the training-patient list during model selection, validation patients do not contribute to the normalization statistics.

In [ ]:
def build_training_memory_safe(
    subjects,
    representation
):

    feature_sum = None
    feature_sq_sum = None

    total_stat_count = 0
    total_windows = 0
    total_pos = 0
    total_neg = 0

    sample_shape = None

    # Pass 1:
    # compute normalization statistics
    # and determine final array size
    for subject in subjects:

        X, y = load_subject(
            subject,
            representation
        )

        if sample_shape is None:

            sample_shape = X.shape[1:]
            feature_dim = X.shape[1]

            feature_sum = np.zeros(
                feature_dim,
                dtype=np.float64
            )

            feature_sq_sum = np.zeros(
                feature_dim,
                dtype=np.float64
            )

        if representation == "raw":

            for channel in range(
                X.shape[1]
            ):

                X_channel = X[
                    :,
                    channel,
                    :
                ].astype(np.float64)

                feature_sum[channel] += (
                    X_channel.sum()
                )

                feature_sq_sum[channel] += (
                    np.square(
                        X_channel
                    ).sum()
                )

                del X_channel

            total_stat_count += (
                X.shape[0]
                * X.shape[2]
            )

        else:

            X_float64 = X.astype(
                np.float64
            )

            feature_sum += (
                X_float64.sum(axis=0)
            )

            feature_sq_sum += (
                np.square(
                    X_float64
                ).sum(axis=0)
            )

            total_stat_count += (
                X.shape[0]
            )

            del X_float64

        total_windows += len(y)

        total_pos += int(
            (y == 1).sum()
        )

        total_neg += int(
            (y == 0).sum()
        )

        del X, y
        gc.collect()

    mean = (
        feature_sum
        / total_stat_count
    )

    variance = (
        feature_sq_sum
        / total_stat_count
        - mean ** 2
    )

    std = np.sqrt(
        np.maximum(
            variance,
            1e-8
        )
    )

    mean = mean.astype(
        np.float32
    )

    std = std.astype(
        np.float32
    )

    # Allocate final arrays once
    X_all = np.empty(
        (
            total_windows,
            *sample_shape
        ),
        dtype=np.float32
    )

    y_all = np.empty(
        total_windows,
        dtype=np.float32
    )

    # Pass 2:
    # fill arrays patient-by-patient
    offset = 0

    for subject in subjects:

        X, y = load_subject(
            subject,
            representation
        )

        n = len(y)

        X_all[
            offset:offset + n
        ] = X

        y_all[
            offset:offset + n
        ] = y

        offset += n

        del X, y
        gc.collect()

    # Normalize in place
    if representation == "raw":

        for channel in range(
            X_all.shape[1]
        ):

            X_all[
                :,
                channel,
                :
            ] -= mean[channel]

            X_all[
                :,
                channel,
                :
            ] /= (
                std[channel]
                + 1e-6
            )

    else:

        X_all -= mean
        X_all /= (
            std + 1e-6
        )

    print(
        f"{representation.upper()} | "
        f"{total_windows} windows | "
        f"seizure={total_pos} | "
        f"normal={total_neg} | "
        f"neg:pos="
        f"{total_neg / total_pos:.1f}:1"
    )

    return (
        X_all,
        y_all,
        mean,
        std
    )

## 5. Patient-Disjoint Epoch Selection

- Models train on the 14 training patients for up to **20 epochs**.
- The four validation patients remain completely excluded from training and normalization.
- After every epoch, AUROC and AUPRC are calculated separately for each validation patient.
- Epoch selection uses the **mean patient-level AUPRC** across the four validation patients.
- AUPRC is used for selection because seizure windows are highly imbalanced relative to normal EEG.

In [ ]:
def prepare_validation(
    representation,
    mean,
    std
):

    cache = {}

    for subject in VAL_SUBJECTS:

        X, y = load_subject(
            subject,
            representation
        )

        X = scale_array(
            X,
            mean,
            std,
            representation
        )

        cache[subject] = (X, y)

    return cache


def select_epoch(representation):

    print(
        "\n" + "=" * 68
    )
    print(
        "SELECTING EPOCH:",
        representation.upper()
    )
    print("=" * 68)

    set_seed()

    X_train, y_train, mean, std = (
        build_training_memory_safe(
            TRAIN_SUBJECTS,
            representation
        )
    )

    val_cache = prepare_validation(
        representation,
        mean,
        std
    )

    model = make_model(
        representation
    )

    train_loader = make_loader(
        X_train,
        y_train,
        True
    )

    criterion = make_loss(
        y_train
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    rows = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        loss = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion
        )

        patient_scores = []

        for subject, (
            X_val,
            y_val
        ) in val_cache.items():

            labels, probs = predict(
                model,
                X_val,
                y_val
            )

            auprc = (
                average_precision_score(
                    labels,
                    probs
                )
            )

            auroc = roc_auc_score(
                labels,
                probs
            )

            patient_scores.append(
                auprc
            )

            rows.append({
                "representation":
                    representation,
                "epoch":
                    epoch,
                "val_subject":
                    subject,
                "auprc":
                    float(auprc),
                "auroc":
                    float(auroc),
            })

        print(
            f"epoch {epoch:02d} | "
            f"loss={loss:.4f} | "
            f"mean patient val AUPRC="
            f"{np.mean(patient_scores):.4f}"
        )

    curve = pd.DataFrame(
        rows
    )

    epoch_summary = (
        curve
        .groupby(
            "epoch",
            as_index=False
        )["auprc"]
        .mean()
        .rename(
            columns={
                "auprc":
                "mean_patient_auprc"
            }
        )
        .sort_values(
            [
                "mean_patient_auprc",
                "epoch"
            ],
            ascending=[
                False,
                True
            ]
        )
    )

    best_epoch = int(
        epoch_summary.iloc[0][
            "epoch"
        ]
    )

    curve.to_csv(
        OUT_DIR
        / f"{representation}_validation_curve.csv",
        index=False
    )

    print(
        f"\nSELECTED "
        f"{representation.upper()} "
        f"EPOCH: {best_epoch}"
    )

    del (
        X_train,
        y_train,
        val_cache,
        model,
        train_loader,
        criterion,
        optimizer
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_epoch

### 5.1 Recorded Epoch Selection

The completed patient-disjoint validation run selected:

- **RAW CNN: epoch 19** — mean validation-patient AUPRC = **0.805849**
- **PSD MLP: epoch 11** — mean validation-patient AUPRC = **0.580667**

The complete per-patient validation curves are saved for later analysis.

In [ ]:
RERUN_SELECTION = False

if RERUN_SELECTION:

    raw_best_epoch = select_epoch(
        "raw"
    )

    psd_best_epoch = select_epoch(
        "psd"
    )

else:

    raw_best_epoch = 19
    psd_best_epoch = 11

    print(
        "Using recorded selections: "
        "RAW=19, PSD=11"
    )

## 6. Additional Held-Out Patient Evaluation

### 6.1 Held-Out Cohort and EEG Preprocessing

- Three additional CHB-MIT patients are evaluated after model development: `chb15`, `chb16`, and `chb18`.
- These patients are **not used for epoch selection or final-model training**.
- Two EDF recordings are used per patient.
- Preprocessing matches the development pipeline: fixed **18-channel montage**, **0.5–50 Hz** filtering, **4-second non-overlapping windows**, and conversion to microvolts.
- A window is positive when at least **2 seconds** overlap a seizure.
- Negative windows must remain at least **60 seconds** away from seizure intervals; ambiguous peri-seizure windows are excluded.

In [ ]:
HELDOUT_SUBJECT_FILES = {
    "chb15": [
        "chb15_06.edf",
        "chb15_10.edf",
    ],
    "chb16": [
        "chb16_10.edf",
        "chb16_11.edf",
    ],
    "chb18": [
        "chb18_29.edf",
        "chb18_30.edf",
    ],
}

STANDARD_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8-0", "P8-O2",
    "FZ-CZ", "CZ-PZ",
]

WINDOW_SECONDS = 4
STEP_SECONDS = 4
EXCLUSION_SECONDS = 60

FREQUENCY_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma": (30, 50),
}

In [ ]:
def extract_log_bandpower(
    X,
    sfreq=256
):

    freqs, psd = welch(
        X,
        fs=sfreq,
        nperseg=256,
        axis=-1
    )

    parts = []

    for low, high in (
        FREQUENCY_BANDS.values()
    ):

        mask = (
            (freqs >= low)
            & (freqs < high)
        )

        power = trapezoid(
            psd[..., mask],
            x=freqs[mask],
            axis=-1
        )

        parts.append(
            np.log10(
                power + 1e-12
            ).astype(np.float32)
        )

    return np.concatenate(
        parts,
        axis=1
    )

### 6.2 Seizure Annotation Parsing

- CHB-MIT summary files provide seizure start and end times for each EDF recording.
- These intervals are parsed once per patient and then used to assign window labels.
- The labeling policy is kept identical to the development data so evaluation uses the same target definition.

In [ ]:
def parse_summary(subject):

    text = (
        DATA_ROOT
        / subject
        / f"{subject}-summary.txt"
    ).read_text(
        errors="ignore"
    )

    seizures = {}
    current = None

    for line in text.splitlines():

        line = line.strip()

        if line.startswith(
            "File Name:"
        ):

            current = (
                line
                .split(":", 1)[1]
                .strip()
            )

        elif (
            current
            and "Seizure" in line
            and "Start Time:" in line
        ):

            start = int(
                re.search(
                    r"(\d+)\s*seconds",
                    line
                ).group(1)
            )

            seizures.setdefault(
                current,
                []
            ).append(
                [start, None]
            )

        elif (
            current
            and "Seizure" in line
            and "End Time:" in line
        ):

            end = int(
                re.search(
                    r"(\d+)\s*seconds",
                    line
                ).group(1)
            )

            seizures[current][-1][1] = end

    return {
        filename: [
            (start, end)
            for start, end
            in intervals
            if end is not None
        ]
        for filename, intervals
        in seizures.items()
    }

In [ ]:
def heldout_labels(
    intervals,
    duration
):

    output = []

    for start in np.arange(
        0,
        duration - WINDOW_SECONDS,
        STEP_SECONDS
    ):

        end = (
            start
            + WINDOW_SECONDS
        )

        overlaps = [
            max(
                0,
                min(
                    end,
                    seizure_end
                )
                - max(
                    start,
                    seizure_start
                )
            )
            for (
                seizure_start,
                seizure_end
            )
            in intervals
        ]

        max_overlap = max(
            overlaps or [0]
        )

        if max_overlap >= 2:

            output.append(
                (start, 1)
            )

            continue

        safely_negative = all(
            end
            <= seizure_start
            - EXCLUSION_SECONDS
            or
            start
            >= seizure_end
            + EXCLUSION_SECONDS

            for (
                seizure_start,
                seizure_end
            )
            in intervals
        )

        if safely_negative:

            output.append(
                (start, 0)
            )

    return output

### 6.3 Held-Out EEG Extraction and Caching

- Each selected EDF is loaded with MNE, restricted to the same 18-channel montage, and filtered from **0.5–50 Hz**.
- EEG amplitudes are converted from volts to microvolts before window extraction.
- The previously defined held-out labeling policy determines which 4-second windows are retained.
- Processed arrays are cached as `.npz` files so EDF loading and filtering do not need to be repeated.
- The cache stores the raw EEG windows and their binary seizure labels for each held-out patient.

In [ ]:
def extract_heldout_subject(subject):

    seizure_map = parse_summary(subject)

    X_parts = []
    y_parts = []

    for filename in (
        HELDOUT_SUBJECT_FILES[subject]
    ):

        print(
            "Reading held-out:",
            subject,
            filename
        )

        raw = mne.io.read_raw_edf(
            DATA_ROOT
            / subject
            / filename,
            preload=True,
            verbose=False
        )

        raw.pick(
            STANDARD_CHANNELS
        )

        raw.filter(
            0.5,
            50.0,
            verbose=False
        )

        sfreq = int(
            raw.info["sfreq"]
        )

        assert sfreq == 256

        data = (
            raw.get_data()
            * 1e6
        ).astype(np.float32)

        intervals = seizure_map.get(
            filename,
            []
        )

        labels = heldout_labels(
            intervals,
            raw.n_times / sfreq
        )

        for start, label in labels:

            start_sample = int(
                start * sfreq
            )

            window = data[
                :,
                start_sample:
                start_sample
                + WINDOW_SECONDS * sfreq
            ]

            if window.shape == (
                18,
                1024
            ):

                X_parts.append(
                    window
                )

                y_parts.append(
                    label
                )

        del raw, data

    return (
        np.stack(
            X_parts
        ).astype(np.float32),

        np.asarray(
            y_parts,
            dtype=np.float32
        )
    )

In [ ]:
def get_heldout_raw():

    output = {}

    for subject in (
        HELDOUT_SUBJECT_FILES
    ):

        cache_path = (
            EXT_CACHE
            / f"{subject}.npz"
        )

        if cache_path.exists():

            with np.load(
                cache_path
            ) as data:

                X = data["X"].astype(
                    np.float32
                )

                y = data["y"].astype(
                    np.float32
                )

            print(
                "Loaded held-out cache:",
                subject
            )

        else:

            print(
                "Extracting held-out EDFs:",
                subject
            )

            X, y = (
                extract_heldout_subject(
                    subject
                )
            )

            np.savez_compressed(
                cache_path,
                X=X,
                y=y
            )

        output[subject] = (
            X,
            y
        )

        print(
            subject,
            "| windows:",
            len(y),
            "| positives:",
            int(y.sum())
        )

    return output

### 6.4 Held-Out RAW vs PSD Evaluation

- Each held-out patient's raw EEG windows are standardized using the **RAW normalization statistics learned from the 18 development patients**.
- The same raw windows are also transformed into 90 PSD features and standardized using the corresponding **PSD development statistics**.
- RAW and PSD therefore evaluate the **same windows and labels** for each patient.
- AUROC and AUPRC are calculated separately for each held-out patient.
- A pooled result is also calculated across all held-out windows, although patient-level results remain the primary descriptive comparison.

In [ ]:
def evaluate_heldout(
    raw_model,
    raw_mean,
    raw_std,
    psd_model,
    psd_mean,
    psd_std,
    heldout_data
):

    rows = []

    all_y = []
    all_raw_probs = []
    all_psd_probs = []

    for subject, (
        X_raw,
        y
    ) in heldout_data.items():

        # RAW representation
        X_raw_scaled = scale_array(
            X_raw,
            raw_mean,
            raw_std,
            "raw"
        )

        raw_labels, raw_probs = predict(
            raw_model,
            X_raw_scaled,
            y
        )

        # PSD representation
        X_psd = extract_log_bandpower(
            X_raw
        )

        X_psd_scaled = scale_array(
            X_psd,
            psd_mean,
            psd_std,
            "psd"
        )

        psd_labels, psd_probs = predict(
            psd_model,
            X_psd_scaled,
            y
        )

        # Both models must evaluate identical labels
        assert np.array_equal(
            raw_labels,
            psd_labels
        )

        rows.append({
            "subject":
                subject,

            "windows":
                len(y),

            "positives":
                int(y.sum()),

            "prevalence":
                float(y.mean()),

            "raw_auroc":
                float(
                    roc_auc_score(
                        y,
                        raw_probs
                    )
                ),

            "raw_auprc":
                float(
                    average_precision_score(
                        y,
                        raw_probs
                    )
                ),

            "psd_auroc":
                float(
                    roc_auc_score(
                        y,
                        psd_probs
                    )
                ),

            "psd_auprc":
                float(
                    average_precision_score(
                        y,
                        psd_probs
                    )
                ),
        })

        all_y.append(y)
        all_raw_probs.append(
            raw_probs
        )
        all_psd_probs.append(
            psd_probs
        )

        del (
            X_raw_scaled,
            X_psd,
            X_psd_scaled
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows)

    # Pooled descriptive result
    y_all = np.concatenate(
        all_y
    )

    raw_probs_all = np.concatenate(
        all_raw_probs
    )

    psd_probs_all = np.concatenate(
        all_psd_probs
    )

    combined_row = {
        "subject":
            "COMBINED",

        "windows":
            len(y_all),

        "positives":
            int(y_all.sum()),

        "prevalence":
            float(y_all.mean()),

        "raw_auroc":
            float(
                roc_auc_score(
                    y_all,
                    raw_probs_all
                )
            ),

        "raw_auprc":
            float(
                average_precision_score(
                    y_all,
                    raw_probs_all
                )
            ),

        "psd_auroc":
            float(
                roc_auc_score(
                    y_all,
                    psd_probs_all
                )
            ),

        "psd_auprc":
            float(
                average_precision_score(
                    y_all,
                    psd_probs_all
                )
            ),
    }

    return pd.concat(
        [
            df,
            pd.DataFrame(
                [combined_row]
            )
        ],
        ignore_index=True
    )

## 7. Final Model Retraining

### 7.1 Retraining on the Full Development Cohort

- After epoch selection is complete, each model is **reinitialized from scratch**.
- RAW and PSD are then retrained using **all 18 development patients**.
- The selected training durations are fixed at **19 epochs for RAW** and **11 epochs for PSD**.
- Normalization statistics and class weights are recomputed using the full 18-patient development cohort.
- No held-out evaluation patient contributes to this final training step.

In [ ]:
def train_final_model_memory_safe(
    representation,
    epochs
):

    print(
        "\n" + "=" * 68
    )

    print(
        f"TRAINING FINAL "
        f"{representation.upper()} "
        f"MODEL FOR {epochs} EPOCHS"
    )

    print("=" * 68)

    set_seed()

    X, y, mean, std = (
        build_training_memory_safe(
            DEV_SUBJECTS,
            representation
        )
    )

    model = make_model(
        representation
    )

    loader = make_loader(
        X,
        y,
        True
    )

    criterion = make_loss(
        y
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    print(
        f"Final training: "
        f"{len(y)} windows | "
        f"positives={int(y.sum())} | "
        f"batches/epoch={len(loader)}"
    )

    for epoch in range(
        1,
        epochs + 1
    ):

        loss = train_epoch(
            model,
            loader,
            optimizer,
            criterion
        )

        print(
            f"final {representation} "
            f"epoch {epoch:02d}/{epochs} | "
            f"loss={loss:.4f}"
        )

    del (
        loader,
        X,
        y,
        criterion,
        optimizer
    )

    gc.collect()

    try:
        ctypes.CDLL(
            "libc.so.6"
        ).malloc_trim(0)

    except Exception:
        pass

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        model,
        mean,
        std
    )

### 7.2 Final Training and Saved Models

- Final training is expensive, so the completed model checkpoints and held-out evaluation results are reused by default.
- Setting `RERUN_FINAL_TRAINING = True` retrains both models from scratch, saves their weights and normalization statistics, and recomputes the held-out evaluation.

In [ ]:
RERUN_FINAL_TRAINING = False

if RERUN_FINAL_TRAINING:

    raw_model, raw_mean, raw_std = (
        train_final_model_memory_safe(
            "raw",
            raw_best_epoch
        )
    )

    psd_model, psd_mean, psd_std = (
        train_final_model_memory_safe(
            "psd",
            psd_best_epoch
        )
    )

    torch.save(
        {
            "model_state_dict":
                raw_model.state_dict(),
            "mean":
                raw_mean,
            "std":
                raw_std,
            "selected_epoch":
                raw_best_epoch,
        },
        OUT_DIR / "final_raw_model.pt"
    )

    torch.save(
        {
            "model_state_dict":
                psd_model.state_dict(),
            "mean":
                psd_mean,
            "std":
                psd_std,
            "selected_epoch":
                psd_best_epoch,
        },
        OUT_DIR / "final_psd_model.pt"
    )

    heldout_data = get_heldout_raw()

    heldout_df = evaluate_heldout(
        raw_model,
        raw_mean,
        raw_std,
        psd_model,
        psd_mean,
        psd_std,
        heldout_data
    )

    heldout_df.to_csv(
        OUT_DIR
        / "external_raw_vs_psd.csv",
        index=False
    )

else:

    heldout_df = pd.read_csv(
        OUT_DIR
        / "external_raw_vs_psd.csv"
    )

    print(
        heldout_df.to_string(
            index=False
        )
    )

## 8. Held-Out Patient Results

The completed final-training run used all **52,271 development windows**, including **660 seizure** and **51,611 normal** windows.

- RAW CNN retrained from scratch for **19 epochs**.
- PSD MLP retrained from scratch for **11 epochs**.
- Performance was then evaluated on the three additional held-out CHB-MIT patients.

| Patient | Windows | Positives | RAW AUROC | RAW AUPRC | PSD AUROC | PSD AUPRC |
|---|---:|---:|---:|---:|---:|---:|
| chb15 | 1,736 | 39 | 0.885862 | 0.243767 | 0.537676 | 0.024357 |
| chb16 | 1,737 | 5 | 0.786605 | 0.206569 | 0.805889 | 0.016697 |
| chb18 | 1,738 | 21 | 0.947833 | 0.690030 | 0.949275 | 0.404304 |

Across patients, RAW achieved a macro mean **AUROC of 0.873433** and **AUPRC of 0.380122**, compared with **0.764280 AUROC** and **0.148453 AUPRC** for PSD.

In [ ]:
patient_results = (
    heldout_df[
        heldout_df["subject"] != "COMBINED"
    ]
    .copy()
)

macro_results = pd.DataFrame({
    "model": [
        "Raw CNN",
        "PSD MLP"
    ],

    "macro_AUROC": [
        patient_results[
            "raw_auroc"
        ].mean(),

        patient_results[
            "psd_auroc"
        ].mean()
    ],

    "macro_AUPRC": [
        patient_results[
            "raw_auprc"
        ].mean(),

        patient_results[
            "psd_auprc"
        ].mean()
    ],

    "median_AUROC": [
        patient_results[
            "raw_auroc"
        ].median(),

        patient_results[
            "psd_auroc"
        ].median()
    ],

    "median_AUPRC": [
        patient_results[
            "raw_auprc"
        ].median(),

        patient_results[
            "psd_auprc"
        ].median()
    ],
})

patient_results[
    "raw_minus_psd_AUROC"
] = (
    patient_results["raw_auroc"]
    - patient_results["psd_auroc"]
)

patient_results[
    "raw_minus_psd_AUPRC"
] = (
    patient_results["raw_auprc"]
    - patient_results["psd_auprc"]
)

print("HELD-OUT PATIENT MACRO RESULTS")
display(macro_results)

print(
    "\nPATIENT-LEVEL RAW - PSD DIFFERENCES"
)

display(
    patient_results[
        [
            "subject",
            "raw_minus_psd_AUROC",
            "raw_minus_psd_AUPRC"
        ]
    ]
)

## 9. Figures

The validation curves show how mean patient-level AUPRC changed across epochs during model selection. Patient-level bar charts summarize the final RAW and PSD performance on the three additional held-out patients.

In [ ]:
raw_curve = pd.read_csv(
    OUT_DIR / "raw_validation_curve.csv"
)

psd_curve = pd.read_csv(
    OUT_DIR / "psd_validation_curve.csv"
)

raw_epoch = (
    raw_curve
    .groupby(
        "epoch",
        as_index=False
    )["auprc"]
    .mean()
    .rename(
        columns={
            "auprc": "mean_patient_auprc"
        }
    )
)

psd_epoch = (
    psd_curve
    .groupby(
        "epoch",
        as_index=False
    )["auprc"]
    .mean()
    .rename(
        columns={
            "auprc": "mean_patient_auprc"
        }
    )
)


plt.figure(
    figsize=(8, 5)
)

plt.plot(
    raw_epoch["epoch"],
    raw_epoch["mean_patient_auprc"],
    marker="o",
    label="Raw CNN"
)

plt.plot(
    psd_epoch["epoch"],
    psd_epoch["mean_patient_auprc"],
    marker="o",
    label="PSD MLP"
)

plt.axvline(
    raw_best_epoch,
    linestyle="--",
    alpha=0.5
)

plt.axvline(
    psd_best_epoch,
    linestyle="--",
    alpha=0.5
)

plt.xlabel("Epoch")
plt.ylabel(
    "Mean Validation-Patient AUPRC"
)

plt.title(
    "Patient-Disjoint Validation Performance"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    OUT_DIR
    / "validation_auprc_by_epoch.png",
    dpi=300
)

plt.show()

In [ ]:
x = np.arange(
    len(patient_results)
)

width = 0.36

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    x - width / 2,
    patient_results["raw_auprc"],
    width,
    label="Raw CNN"
)

plt.bar(
    x + width / 2,
    patient_results["psd_auprc"],
    width,
    label="PSD MLP"
)

plt.xticks(
    x,
    patient_results["subject"]
)

plt.xlabel("Held-Out Patient")
plt.ylabel("AUPRC")

plt.title(
    "Held-Out Patient-Level AUPRC"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    OUT_DIR
    / "heldout_patient_auprc.png",
    dpi=300
)

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.bar(
    x - width / 2,
    patient_results["raw_auroc"],
    width,
    label="Raw CNN"
)

plt.bar(
    x + width / 2,
    patient_results["psd_auroc"],
    width,
    label="PSD MLP"
)

plt.xticks(
    x,
    patient_results["subject"]
)

plt.xlabel("Held-Out Patient")
plt.ylabel("AUROC")

plt.title(
    "Held-Out Patient-Level AUROC"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    OUT_DIR
    / "heldout_patient_auroc.png",
    dpi=300
)

plt.show()

## 10. Interpretation

Patient-disjoint model selection chose **19 epochs for the Raw CNN** and **11 epochs for the PSD MLP**, based on mean AUPRC across four held-out validation patients.

After retraining from scratch on all 18 development patients, the Raw CNN showed stronger precision-recall performance on each of the three additional held-out patients. RAW achieved a macro patient-level AUPRC of **0.380**, compared with **0.148** for PSD.

AUROC showed a less uniform pattern. RAW performed substantially better on `chb15`, while PSD achieved slightly higher AUROC on `chb16` and `chb18`. This illustrates why patient-level performance should be examined rather than relying only on a pooled metric.

Taken together, the results are consistent with the raw EEG representation retaining useful temporal waveform information that is not fully preserved by the compressed five-band spectral representation. This should be interpreted as a representation-specific result under the models and preprocessing used here, rather than evidence that raw EEG is universally superior.

## 11. Limitations

- Epoch selection uses a single fixed **14-patient training / 4-patient validation split**, so the selected training duration may vary under a different patient split.
- The additional held-out evaluation contains only **three patients**, so these results are descriptive rather than a basis for statistical inference.
- The held-out patients come from the same **CHB-MIT dataset**, not an independent clinical dataset.
- These patients had been inspected during earlier project iterations, so the final evaluation should not be interpreted as a pristine confirmatory test set.
- Only a subset of recordings is used for each patient rather than the complete CHB-MIT corpus.
- RAW and PSD use different model architectures because their input structures differ.
- The results demonstrate patient-independent classification within this experimental setting and do not constitute clinical validation.